In [ ]:
from pathlib import Path
import sys

import pandas as pd
import networkx as nx

PROJECT_ROOT = Path("..")
SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

from load_data import load_hpo, load_hpo_annotations, get_hpo_labels
from build_graph import (
    filter_diseases_by_annotaion_count, 
    build_disease_phenotype_graph, 
    add_hpo_hierarchy_edges, 
    get_graph_statistics, 
    export_graph_edges, 
    export_graph_nodes)

In [3]:
HPO_PATH = PROJECT_ROOT / "data" / "raw" / "hp.obo"
HPOA_PATH = PROJECT_ROOT / "data" / "raw" / "phenotype.hpoa"

hpo = load_hpo(HPO_PATH)
hpoa = load_hpo_annotations(HPOA_PATH)
hpo_labels = get_hpo_labels(hpo)

print("HPO nodes:", len(hpo.nodes))
print("HPO edges:", len(hpo.edges))
print("Annotation rows:", len(hpoa))
print("Diseases:", hpoa["database_id"].nunique())
print("HPO terms:", hpoa["hpo_id"].nunique())

HPO nodes: 19389
HPO edges: 23677
Annotation rows: 264239
Diseases: 12971
HPO terms: 11514


In [5]:
hpoa_filtered = filter_diseases_by_annotaion_count(
    hpoa,
    min_annotations=5,
    max_annotations=50,
    max_diseases=1000,
    random_state=42,
)

print("Filtered rows:", len(hpoa_filtered))
print("Filtered diseases", hpoa_filtered["database_id"].nunique())
print("Filtered HPO terms:", hpoa_filtered["hpo_id"].nunique())

hpoa_filtered.head()

Filtered rows: 20178
Filtered diseases 1000
Filtered HPO terms: 4569


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [6]:
G_dp = build_disease_phenotype_graph(hpoa_filtered)

stats_dp = get_graph_statistics(G_dp)
stats_dp

{'nodes_total': 5569,
 'edges_total': 20132,
 'nodes_disease': 1000,
 'nodes_phenotype': 4569,
 'edges_has_phenotype': 20132}

In [7]:
G_dph = build_disease_phenotype_graph(hpoa_filtered)
G_dph = add_hpo_hierarchy_edges(G_dph, hpo)

stats_dph = get_graph_statistics(G_dph)
stats_dph

{'nodes_total': 11760,
 'edges_total': 31521,
 'nodes_disease': 1000,
 'nodes_phenotype': 10760,
 'edges_has_phenotype': 20132,
 'edges_is_a': 11389}

In [10]:
stats_df = pd.DataFrame(
    [
        {"graph": "disease_phenotype", **stats_dp},
        {"graph": "disease_phenotype_hpo_hirachy", **stats_dph},
    ]
)

stats_df

,graph,nodes_total,edges_total,nodes_disease,nodes_phenotype,edges_has_phenotype,edges_is_a
0,disease_phenotype,5569,20132,1000,4569,20132,NaN
1,disease_phenotype_hpo_hirachy,11760,31521,1000,10760,20132,11389.0


In [12]:
disease_nodes = [
    node
    for node, data in G_dph.nodes(data=True)
    if data.get("node_type") == "disease"
]

disease_nodes[:10]

example_disease = disease_nodes[0]
print("Example disease:", example_disease)
print("Label:", G_dph.nodes[example_disease].get("label"))

neighbors = list(G_dph.neighbors(example_disease))

for neighbor in neighbors[:20]:
    print(neighbor, G_dph.nodes[neighbor].get("label"))

Example disease: OMIM:619340
Label: Developmental and epileptic encephalopathy 96
HP:0011097 Epileptic spasm
HP:0002187 Profound intellectual disability
HP:0001518 Small for gestational age
HP:0032792 Tonic seizure
HP:0011451 Primary microcephaly
HP:0010851 EEG with burst suppression
HP:0001789 Hydrops fetalis
HP:0200134 Epileptic encephalopathy
HP:0002643 Neonatal respiratory distress


In [13]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

export_graph_nodes(
    G_dp,
    PROCESSED_DIR / "nodes_disease_phenotype.csv",
)

export_graph_edges(
    G_dp,
    PROCESSED_DIR / "edges_disease_phenotype.csv",
)

export_graph_nodes(
    G_dph,
    PROCESSED_DIR / "nodes_disease_phenotype_hpo_hierarchy.csv",
)

export_graph_edges(
    G_dph,
    PROCESSED_DIR / "edges_disease_phenotype_hpo_hierarchy.csv",
)

hpoa_filtered.to_csv(
    PROCESSED_DIR / "hpoa_filtered.csv",
    index=False,
)

stats_df.to_csv(
    PROCESSED_DIR / "graph_statistics.csv",
    index=False,
)